# Rich Display with Fortran

Jupyter can render HTML, mathematical notation, vector graphics, and pixel images produced by Fortran. The examples below use the `lfortran_display` module to create and display these outputs.


## 1 — Reusable rich values

Create a formatted value, store it in a variable, and enter the variable name whenever you want to display it.


In [ ]:
use lfortran_display, only: mime_bundle
character(len=:), allocatable :: summary
summary = mime_bundle('text/html', &
  '<div style="padding:16px;border-left:4px solid #2563eb;font-family:sans-serif">' // &
  '<b>Quarterly summary</b><br>Revenue increased by <strong>18%</strong>.</div>')


In [ ]:
summary

In [ ]:
character(len=:), allocatable :: svg, image
svg = "<svg xmlns='http://www.w3.org/2000/svg' width='540' height='260'" // &
      " style='background:white;font-family:sans-serif'>" // &
      "<text x='270' y='28' text-anchor='middle' font-size='18'>Sample measurements</text>" // &
      "<line x1='55' y1='215' x2='510' y2='215' stroke='#94a3b8'/>" // &
      "<line x1='55' y1='45' x2='55' y2='215' stroke='#94a3b8'/>" // &
      "<polyline points='55,190 125,155 195,170 265,105 335,125 405,70 475,88'" // &
      " fill='none' stroke='#2563eb' stroke-width='4'/>" // &
      "<g fill='#f97316' stroke='white' stroke-width='2'>" // &
      "<circle cx='55' cy='190' r='6'/><circle cx='125' cy='155' r='6'/>" // &
      "<circle cx='195' cy='170' r='6'/><circle cx='265' cy='105' r='6'/>" // &
      "<circle cx='335' cy='125' r='6'/><circle cx='405' cy='70' r='6'/>" // &
      "<circle cx='475' cy='88' r='6'/></g>" // &
      "<text x='282' y='245' text-anchor='middle' fill='#475569'>Time</text>" // &
      "</svg>"
image = mime_bundle('image/svg+xml', svg)


In [ ]:
image

The SVG is stored in `image`. Entering the variable name displays it again.


In [ ]:
image

## 2 — Displaying multiple outputs

A cell can also display several results as it runs.


In [ ]:
use lfortran_display, only: display_data
call display_data('text/html', '<p><b>A formatted message</b></p>')
call display_data('text/latex', '$$\int_{-\infty}^{\infty} e^{-x^2}\,dx = \sqrt{\pi}$$')


## 3 — Pixel image: HSV colour wheel

This example constructs a bitmap entirely in Fortran, encodes it as BMP, and displays it in the notebook. White pixels outside the unit circle provide the background.


In [ ]:
! Inline BMP encoder (copy from Mandelbrot.ipynb or bmp_display.f90)
module bmp_mod
  use lfortran_display, only: display_data
  implicit none
contains
  subroutine display_image_bmp(w, h, rgba)
    integer, intent(in) :: w, h, rgba(4,w,h)
    integer :: rs, pad, fs, r, c, idx
    character(len=:), allocatable :: bmp
    rs = w*3; pad = mod(4-mod(rs,4),4); fs = 54+(rs+pad)*h
    allocate(character(len=fs) :: bmp); bmp = repeat(char(0),fs)
    bmp(1:2) = 'BM'
    call le32(bmp,3,fs); call le32(bmp,11,54)
    call le32(bmp,15,40); call le32(bmp,19,w); call le32(bmp,23,h)
    call le16(bmp,27,1); call le16(bmp,29,24)
    idx = 55
    do r = h, 1, -1
      do c = 1, w
        bmp(idx:idx)     = char(min(255,max(0,rgba(3,c,r))))
        bmp(idx+1:idx+1) = char(min(255,max(0,rgba(2,c,r))))
        bmp(idx+2:idx+2) = char(min(255,max(0,rgba(1,c,r))))
        idx = idx + 3
      end do
      idx = idx + pad
    end do
    call display_data('image/bmp', b64(bmp))
  end subroutine
  subroutine le32(s,o,v); character(len=*),intent(inout)::s; integer,intent(in)::o,v
    s(o:o)=char(iand(v,255)); s(o+1:o+1)=char(iand(ishft(v,-8),255))
    s(o+2:o+2)=char(iand(ishft(v,-16),255)); s(o+3:o+3)=char(iand(ishft(v,-24),255))
  end subroutine
  subroutine le16(s,o,v); character(len=*),intent(inout)::s; integer,intent(in)::o,v
    s(o:o)=char(iand(v,255)); s(o+1:o+1)=char(iand(ishft(v,-8),255))
  end subroutine
  function b64(data) result(out)
    character(len=*),intent(in)::data; character(len=:),allocatable::out
    character(len=64),parameter::alpha='ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/'
    integer::n,i,j,v
    n=len(data); allocate(character(len=((n+2)/3)*4)::out); j=1
    do i=1,n,3
      v=ishft(ichar(data(i:i)),16)
      if(i+1<=n) v=v+ishft(ichar(data(i+1:i+1)),8)
      if(i+2<=n) v=v+ichar(data(i+2:i+2))
      out(j:j)=alpha(ishft(v,-18)+1:ishft(v,-18)+1)
      out(j+1:j+1)=alpha(iand(ishft(v,-12),63)+1:iand(ishft(v,-12),63)+1)
      out(j+2:j+2)=merge(alpha(iand(ishft(v,-6),63)+1:iand(ishft(v,-6),63)+1),'=',i+1<=n)
      out(j+3:j+3)=merge(alpha(iand(v,63)+1:iand(v,63)+1),'=',i+2<=n)
      j=j+4
    end do
  end function
end module bmp_mod

In [ ]:
use bmp_mod
integer, parameter :: W = 256, H = 256
integer :: pixels(4,W,H), ci, cj, hi
real :: cx, cy, dist, hue, r, g, b, sat, val, f, p, q, t

do cj = 1, H
  do ci = 1, W
    cx = (ci - W/2.0) / (W/2.0)
    cy = (cj - H/2.0) / (H/2.0)
    dist = sqrt(cx**2 + cy**2)
    if (dist > 1.0) then
      pixels(:,ci,cj) = [255, 255, 255, 255]  ! white outside circle
    else
      hue = atan2(cy, cx) / (2.0 * 3.14159265) + 0.5
      sat = dist; val = 1.0
      hi = int(hue * 6)
      f = hue*6 - hi; p = val*(1-sat); q = val*(1-f*sat); t = val*(1-(1-f)*sat)
      if     (mod(hi,6)==0) then; r=val; g=t;   b=p
      else if(mod(hi,6)==1) then; r=q;   g=val; b=p
      else if(mod(hi,6)==2) then; r=p;   g=val; b=t
      else if(mod(hi,6)==3) then; r=p;   g=q;   b=val
      else if(mod(hi,6)==4) then; r=t;   g=p;   b=val
      else;                       r=val; g=p;   b=q
      end if
      pixels(1,ci,cj)=int(r*255); pixels(2,ci,cj)=int(g*255)
      pixels(3,ci,cj)=int(b*255); pixels(4,ci,cj)=255
    end if
  end do
end do
call display_image_bmp(W, H, pixels)

## 4 — Clear output

`clear_output()` removes output produced earlier by the current cell.


In [ ]:
use lfortran_display
call display_data('text/html', '<p>This output is cleared.</p>')
call clear_output()
call display_data('text/html', '<p><b>Only this output remains.</b></p>')